In [ ]:
# importting libraries
import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
import re
from sklearn.preprocessing import LabelEncoder

In [10]:
# Read Dataset
df = pd.read_csv("./data/fake_or_real_news.csv")
df.head(10)

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
5,6903,"Tehran, USA","\nI’m not an immigrant, but my grandparents ...",FAKE
6,7341,Girl Horrified At What She Watches Boyfriend D...,"Share This Baylee Luciani (left), Screenshot o...",FAKE
7,95,‘Britain’s Schindler’ Dies at 106,A Czech stockbroker who saved more than 650 Je...,REAL
8,4869,Fact check: Trump and Clinton at the 'commande...,Hillary Clinton and Donald Trump made some ina...,REAL
9,2909,Iran reportedly makes new push for uranium con...,Iranian negotiators reportedly have made a las...,REAL


In [11]:
df.isnull().sum() # che dataset tamizi!

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [12]:
# Encoding the Labels
df["label"] = LabelEncoder().fit_transform(df["label"])
df["label"]

0       0
1       0
2       1
3       0
4       1
       ..
6330    1
6331    0
6332    0
6333    1
6334    1
Name: label, Length: 6335, dtype: int64

### Steaming:
Steaming is the process of reducing a word to its root form. For example, the words "running", "runner", and "ran" can all be reduced to the root word "run". This is done to help improve the performance of natural language processing (NLP) models by reducing the number of unique words in the dataset.

In [4]:
# download stopwords
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\moham\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
df["content"] = df["title"] + " " + df["text"]

df['content']= df['content'].apply(str)

In [15]:
Porter = PorterStemmer()

# Pre-compile regex and load stopwords once
REGEX_PATTERN = re.compile("[^a-zA-Z]")
STOP_WORDS = set(stopwords.words("english"))

# optimized steaming function
def stemming(content):
    stemmed_content = REGEX_PATTERN.sub(" ", content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [
        Porter.stem(word) for word in stemmed_content if word not in STOP_WORDS
    ]
    stemmed_content = " ".join(stemmed_content)
    return stemmed_content


In [16]:

df['content'] = df['content'].apply(stemming)

print("Stemming completed!")


Stemming completed!


In [18]:
X = df["content"].values
Y = df["label"].values


In [19]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1608995 stored elements and shape (6335, 43893)>

In [20]:
# splitting train and test data

Xtr , Xte , Ytr , Yte = train_test_split(X , Y , train_size=0.8 , stratify=Y , random_state=42)


In [21]:
# train model 
model = LogisticRegression()
model.fit(Xtr , Ytr)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [22]:
# Evaluation

train_acc = accuracy_score(model.predict(Xtr) , Ytr)
print(f"train Accuracy: {train_acc}")

train Accuracy: 0.9514601420678769


In [23]:
# test data Evaluation

test_acc = accuracy_score(model.predict(Xte) , Yte)
print(f"test Accuracy: {test_acc}")

test Accuracy: 0.9210734017363852


test the model on a new news


In [33]:
new = Xte[1]

prediction = model.predict(new)
print(prediction)

print("The News is Real") if prediction[0] == 1 else print("The News is Fake")

print(f"\n {Yte[1]} is the actual label")

[0]
The News is Fake

 0 is the actual label
